In [1]:
# using this 

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import html

# --- STEP 1: THE SCRAPER ---
class MedicalDatasetBuilder:
    def __init__(self):
        self.headers = {'User-Agent': 'HealthResearchBot/1.0'}
        self.data = []
        
        self.targets = {
            "kanker.nl": {"cat": "Oncology", "sel": "main", "urls": ["https://www.kanker.nl/bibliotheek", "https://www.kanker.nl/gevolgen-van-kanker"]},
            "thuisarts.nl": {"cat": "General Medical", "sel": "article", "urls": ["https://www.thuisarts.nl/onderwerpen"]},
            "iknl.nl": {"cat": "Research", "sel": ".content-block", "urls": ["https://iknl.nl/nieuws", "https://iknl.nl/over-iknl"]},
            "nfk.nl": {"cat": "Patient Fed", "sel": ".content", "urls": ["https://nfk.nl/over-kanker"]},
            "zorgwijzer.nl": {"cat": "Policy/Insurance", "sel": ".article-content", "urls": ["https://www.zorgwijzer.nl/zorgverzekering-2026"]},
            "zorginstituutnederland.nl": {"cat": "Policy", "sel": "#content", "urls": ["https://www.zorginstituutnederland.nl/over-ons"]}
        }

    def scrape_all(self):
        for domain, info in self.targets.items():
            for url in info['urls']:
                try:
                    res = requests.get(url, headers=self.headers, timeout=10)
                    soup = BeautifulSoup(res.text, 'html.parser')
                    title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No Title"
                    content_node = soup.select_one(info['sel'])
                    content = content_node.get_text(separator=' ', strip=True) if content_node else ""

                    self.data.append({
                        "Domain": domain,
                        "Category": info['cat'],
                        "Title": title,
                        "Body": content,
                        "URL": url
                    })
                    print(f"Scraped: {domain} | {title[:30]}...")
                    time.sleep(1)
                except Exception as e:
                    print(f"Error {domain}: {e}")

    def get_dataframe(self):
        return pd.DataFrame(self.data)

# --- STEP 2: THE AI CLEANER ---
class MedicalDataCleaner:
    def __init__(self, raw_df):
        self.df = raw_df.copy()
        
    def clean_text(self, text):
        if not isinstance(text, str): return ""
        text = html.unescape(text)
        text = re.sub(r'http\S+|[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        boilerplate = ["alle rechten voorbehouden", "cookie-instellingen", "lees meer"]
        for phrase in boilerplate:
            text = text.replace(phrase, "")
        return text

    def chunk_text(self, text, size=800):
        words = text.split()
        return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

    def run_pipeline(self):
        self.df['Body_Clean'] = self.df['Body'].apply(self.clean_text)
        self.df['Title_Clean'] = self.df['Title'].apply(self.clean_text)
        
        kb_table = self.df[['Domain', 'Category', 'Title_Clean', 'Body_Clean', 'URL']]
        meta_table = self.df[['Domain', 'Category']].drop_duplicates().reset_index(drop=True)
        meta_table['Trust_Level'] = "Official/Verified"
        
        chunks = []
        for _, row in self.df.iterrows():
            text_chunks = self.chunk_text(row['Body_Clean'])
            for i, chunk in enumerate(text_chunks):
                chunks.append({
                    "chunk_id": f"{row['Domain']}_{i}",
                    "parent_url": row['URL'],
                    "text": chunk,
                    "category": row['Category']
                })
        chunks_table = pd.DataFrame(chunks)
        
        return kb_table, meta_table, chunks_table

# --- STEP 3: EXECUTION ---

# 1. Scrape
builder = MedicalDatasetBuilder()
builder.scrape_all()
raw_df = builder.get_dataframe()

# 2. Clean
cleaner = MedicalDataCleaner(raw_df)
df_kb, df_meta, df_chunks = cleaner.run_pipeline()

# 3. View All Data
# Setting pandas to show all text content without clipping
pd.set_option('display.max_colwidth', None)

print("\n" + "="*50)
print("TABLE 1: KNOWLEDGE BASE (All Rows)")
print("="*50)
print(df_kb)

print("\n" + "="*50)
print("TABLE 2: METADATA & SOURCE TRUST")
print("="*50)
print(df_meta)

print("\n" + "="*50)
print("TABLE 3: CHUNKS FOR LLM (First 5 Rows)")
print("="*50)
print(df_chunks.head())

# Optional: Save to CSVs to inspect locally
# df_kb.to_csv("clean_medical_kb.csv", index=False)

Scraped: kanker.nl | Betrouwbare informatie...
Scraped: kanker.nl | Gevolgen van kanker...
Scraped: thuisarts.nl | Deze pagina bestaat niet meer...
Scraped: iknl.nl | Nieuws...
Scraped: iknl.nl | Over IKNL...
Scraped: nfk.nl | 404...
Scraped: zorgwijzer.nl | Zorgverzekering2026 vergelijke...
Scraped: zorginstituutnederland.nl | Wat wij doen...

TABLE 1: KNOWLEDGE BASE (All Rows)
                      Domain          Category  \
0                  kanker.nl          Oncology   
1                  kanker.nl          Oncology   
2               thuisarts.nl   General Medical   
3                    iknl.nl          Research   
4                    iknl.nl          Research   
5                     nfk.nl       Patient Fed   
6              zorgwijzer.nl  Policy/Insurance   
7  zorginstituutnederland.nl            Policy   

                       Title_Clean  \
0           Betrouwbare informatie   
1              Gevolgen van kanker   
2    Deze pagina bestaat niet meer   
3              

In [2]:
# Create a specialized table for CSN Question Analysis
csn_analysis_df = df_kb.copy()

# 1. Action Tagging
def auto_tag(text):
    tags = []
    if "wachttijd" in text.lower(): tags.append("Waiting Times")
    if "verwijzing" in text.lower(): tags.append("Referrals")
    if "second opinion" in text.lower(): tags.append("Second Opinion")
    return ", ".join(tags)

csn_analysis_df['Action_Tags'] = csn_analysis_df['Body_Clean'].apply(auto_tag)

# 2. Stakeholder Identification
csn_analysis_df['Stakeholder'] = csn_analysis_df['Body_Clean'].apply(
    lambda x: "Caregiver" if "naaste" in x.lower() or "partner" in x.lower() else "Patient"
)

# 3. Refresh Tracking
csn_analysis_df['Scrape_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d')

In [3]:
import pandas as pd
import IPython.display as display # Use this if you are in a Jupyter Notebook

class DatasetVisualizer:
    @staticmethod
    def show_all(df_kb, df_meta, df_chunks):
        # 1. Terminal/Notebook Formatting
        pd.set_option('display.max_columns', None)
        pd.set_option('display.max_rows', 10)  # Shows top/bottom 5 to keep it clean
        pd.set_option('display.max_colwidth', 50) # Keeps body text from taking up the whole screen

        print("\n" + "█"*60)
        print(" VIEWING: KNOWLEDGE BASE (Primary Journey Data)")
        print("█"*60)
        print(df_kb[['Category', 'Journey_Stage', 'Action_Tags', 'Title_Clean']].head(10))

        print("\n" + "█"*60)
        print(" VIEWING: SOURCE FRESHNESS METADATA")
        print("█"*60)
        print(df_meta)

    @staticmethod
    def export_for_audit(df_kb):
        """Saves a CSV that you can open in Excel to manually check for gaps."""
        filename = "navigator_audit_log.csv"
        df_kb.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n✅ Audit dataset saved to: {filename}")

# --- ENHANCED CLEANER WITH JOURNEY TAGS ---
class StrategicCleaner(MedicalDataCleaner):
    def run_strategic_pipeline(self):
        # Trigger the base cleaning first
        kb_raw, meta, chunks = self.run_pipeline()
        
        # FIX: Create an explicit copy so we don't trigger the SettingWithCopyWarning
        kb = kb_raw.copy()
        
        # Adding Journey Stages based on keywords in Body
        def get_journey_stage(text):
            text = text.lower()
            if any(w in text for w in ["onderzoek", "uitslag", "biopt"]): return "Diagnosis"
            if any(w in text for w in ["bestraling", "chemo", "operatie"]): return "Treatment"
            if any(w in text for w in ["nazorg", "herstel", "controle"]): return "Aftercare"
            return "General Information"

        def get_action_tag(text):
            text = text.lower()
            tags = []
            if "wachttijd" in text: tags.append("Waiting Times")
            if "verwijzing" in text: tags.append("Referrals")
            if "second opinion" in text: tags.append("Decision Support")
            return ", ".join(tags) if tags else "General"

        # Now we can safely set these columns using .loc
        kb.loc[:, 'Journey_Stage'] = kb['Body_Clean'].apply(get_journey_stage)
        kb.loc[:, 'Action_Tags'] = kb['Body_Clean'].apply(get_action_tag)
        
        return kb, meta, chunks

# --- EXECUTION ---
cleaner = StrategicCleaner(raw_df)
df_kb, df_meta, df_chunks = cleaner.run_strategic_pipeline()

# Visualizing
viz = DatasetVisualizer()
viz.show_all(df_kb, df_meta, df_chunks)
viz.export_for_audit(df_kb)


████████████████████████████████████████████████████████████
 VIEWING: KNOWLEDGE BASE (Primary Journey Data)
████████████████████████████████████████████████████████████
           Category        Journey_Stage Action_Tags  \
0          Oncology            Diagnosis     General   
1          Oncology  General Information     General   
2   General Medical  General Information     General   
3          Research  General Information     General   
4          Research  General Information     General   
5       Patient Fed  General Information     General   
6  Policy/Insurance  General Information     General   
7            Policy  General Information     General   

                       Title_Clean  
0           Betrouwbare informatie  
1              Gevolgen van kanker  
2    Deze pagina bestaat niet meer  
3                           Nieuws  
4                        Over IKNL  
5                              404  
6  Zorgverzekering2026 vergelijken  
7                     Wat wi